## Импорт датасета 
Импортирую датасеты с HuggingFace


In [ ]:
!pip install datasets pandas Pillow matplotlib albumentations opencv-python

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from datasets import load_dataset
import albumentations as A
import cv2
import warnings

warnings.filterwarnings("ignore") # Отключаем предупреждения matplotlib

# 1. Настройки
NUM_SAMPLES = 100000       # Сколько картинок генерировать из каждого датасета
IMG_DIR = "/content/dataset_images"
CSV_PATH = "/content/data.csv"

os.makedirs(IMG_DIR, exist_ok=True)

# 2. Настраиваем аугментацию (Albumentations)
augmentor = A.Compose([
    A.Rotate(limit=3, border_mode=cv2.BORDER_CONSTANT, value=(255,255,255), p=0.5), # Легкий поворот (как кривой скан)
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),                                    # Шум камеры
    A.MotionBlur(blur_limit=3, p=0.3),                                              # Смаз при движении
    A.ImageCompression(quality_lower=40, quality_upper=80, p=0.5),                  # JPEG артефакты
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3)                              # Перепады освещения
])

# 3. Функции рендеринга
def render_russian_text(text, img_size=(512, 128)):
    """Рендерит обычный текст в картинку с помощью Pillow"""
    img = Image.new('RGB', img_size, color='white')
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()

    # Центрируем текст приблизительно
    draw.text((20, 40), text, fill='black', font=font)
    return np.array(img)

def render_latex(formula, img_size=(512, 128)):
    """Пытается отрендерить LaTeX как формулу через Matplotlib"""
    fig, ax = plt.subplots(figsize=(img_size[0]/100, img_size[1]/100), dpi=100)
    ax.axis('off')
    try:
        # mathtext в matplotlib поддерживает базовый LaTeX
        ax.text(0.5, 0.5, f"${formula}$", size=22, ha='center', va='center')
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        plt.close(fig)
        return img
    except Exception:
        plt.close(fig)
        # Fallback: если макрос сложный, рендерим просто как текст (сырой код)
        return render_russian_text(formula, img_size)

# Списки для сборки data.csv
img_paths = []
texts = []
img_counter = 0

# 4. Загрузка и обработка Русского датасета (GSM8K-ru)
print("Обработка русского датасета...")
ds_ru = load_dataset("d0rj/gsm8k-ru", split="train")

for i, row in enumerate(ds_ru):
    if i >= NUM_SAMPLES: break

    # Убираем переносы строк, чтобы не ломать CSV. Длину можно увеличить (например, до 120-140 символов)
    text = row['question'].replace('\n', ' ').replace('\r', ' ')

    image_np = render_russian_text(text)
    augmented = augmentor(image=image_np)['image']

    img_filename = f"ru_{img_counter}.jpg"
    save_path = os.path.join(IMG_DIR, img_filename)
    Image.fromarray(augmented).save(save_path)

    img_paths.append(save_path)
    texts.append(text)
    img_counter += 1

# 5. Загрузка и обработка LaTeX датасета
print("Обработка LaTeX датасета...")
ds_latex = load_dataset("OleehyO/latex-formulas", "cleaned_formulas", split="train")

for i, row in enumerate(ds_latex):
    if i >= NUM_SAMPLES: break

    # Также очищаем формулу от переносов строк внутри кода
    formula = row['latex_formula'].replace('\n', ' ').replace('\r', ' ')

    image_pil = row['image'].convert('RGB').resize((512, 128))
    image_np = np.array(image_pil)

    augmented = augmentor(image=image_np)['image']

    img_filename = f"latex_{img_counter}.jpg"
    save_path = os.path.join(IMG_DIR, img_filename)
    Image.fromarray(augmented).save(save_path)

    img_paths.append(save_path)
    texts.append(formula)
    img_counter += 1

# 6. Создание data.csv
print("Создание data.csv...")
df = pd.DataFrame({
    'img_path': img_paths,
    'text': texts
})

# Перемешаем датасет, чтобы латех и русский текст шли вперемешку
df = df.sample(frac=1).reset_index(drop=True)
df.to_csv(CSV_PATH, index=False)

print(f"Готово! Сгенерировано {len(df)} картинок. Путь к CSV: {CSV_PATH}")

In [ ]:
import pandas as pd
import re
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

df = pd.read_csv("/content/data.csv")

# Ваша регулярка для команд
latex_command_regex = re.compile(r'\\[a-zA-Z]+')

alphabet = set()

for text in df['text']:
    text = str(text)
    # 1. Достаем команды
    commands = latex_command_regex.findall(text)
    alphabet.update(commands)

    # 2. Достаем все остальные символы
    text_without_commands = latex_command_regex.sub('', text)
    alphabet.update(list(text_without_commands))

# Формируем итоговый словарь
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
special_tokens = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

vocab = special_tokens + sorted(list(alphabet))
char2idx = {token: idx for idx, token in enumerate(vocab)}
idx2char = {idx: token for idx, token in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
print(f"Размер словаря: {VOCAB_SIZE}")
import torchvision.transforms.functional as F

class PadResize:
    def __init__(self, target_h=128, target_w=512):
        self.target_h = target_h
        self.target_w = target_w

    def __call__(self, img):
        # img - PIL Image
        w, h = img.size
        # Считаем коэффициент масштабирования, чтобы вписать картинку
        scale = min(self.target_w / w, self.target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)

        img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)

        # Создаем белую подложку нужного размера
        new_img = Image.new('RGB', (self.target_w, self.target_h), color='white')
        # Вставляем отмасштабированную картинку в центр (или в верхний левый угол)
        new_img.paste(img, (0, 0))
        return new_img

# Замените Resize в вашем датасете на это:

class LatexOCRDataset(Dataset):
    def __init__(self, df, char2idx, max_len=150, transform=None):
        self.df = df
        self.char2idx = char2idx
        self.max_len = max_len
        self.transform = transforms.Compose([
        PadResize(128, 512),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.df)

    def text_to_tensor(self, text):
        # Разбиваем текст на токены: либо команда (\frac), либо один любой символ (.)
        # re.DOTALL позволяет захватывать переносы строк, если они есть
        raw_tokens = re.findall(r'\\[a-zA-Z]+|.', str(text), re.DOTALL)

        tokens = [self.char2idx.get(t, UNK_IDX) for t in raw_tokens]
        tokens = [SOS_IDX] + tokens + [EOS_IDX]

        if len(tokens) < self.max_len:
            tokens.extend([PAD_IDX] * (self.max_len - len(tokens)))
        else:
            tokens = tokens[:self.max_len-1] + [EOS_IDX]

        return torch.tensor(tokens, dtype=torch.long)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = Image.open(row['img_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (512, 128), color='white')

        if self.transform:
            image = self.transform(image)

        text_tensor = self.text_to_tensor(row['text'])
        return image, text_tensor

# Для честной оценки метрик обязательно бьем датасет на train/val
train_df = df.sample(frac=0.9, random_state=42)
val_df = df.drop(train_df.index)

train_dataset = LatexOCRDataset(train_df, char2idx)
val_dataset = LatexOCRDataset(val_df, char2idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

## Модели PyTorch
Задаю две модели для свертки и генерации текста, объединяю в единную модель

In [ ]:
from torchvision import transforms, models
import torch.nn as nn
import math
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) # Shape: (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        resnet = models.resnet18(pretrained=True)

        # Модифицируем слои, чтобы они не сжимали по ширине!
        # layer3 и layer4 по умолчанию имеют stride=(2, 2)
        resnet.layer3[0].conv1.stride = (2, 1) # Сжимаем высоту в 2 раза, ширину не трогаем
        resnet.layer3[0].downsample[0].stride = (2, 1)

        resnet.layer4[0].conv1.stride = (2, 1)
        resnet.layer4[0].downsample[0].stride = (2, 1)

        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.proj = nn.Conv2d(512, d_model, kernel_size=1)

    def forward(self, x):
        features = self.backbone(x)
        # Теперь размер будет не (B, 512, 4, 16), а (B, 512, 4, 64) -> 256 визуальных токенов!
        features = self.proj(features)
        features = features.flatten(2).permute(0, 2, 1)
        return features

class OCRTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nheads=8, num_layers=4, max_len=150):
        super().__init__()
        self.d_model = d_model

        self.encoder = CNNEncoder(d_model)

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nheads, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        # Маска, чтобы модель не "подглядывала" в будущее при обучении
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, images, texts):
        # 1. Извлекаем фичи картинки (Encoder)
        memory = self.encoder(images) # (B, Seq_img, d_model)

        # 2. Подготавливаем текст (Decoder input)
        # Убираем последний токен текста для входа, чтобы предсказывать сдвинутую последовательность
        tgt = self.embedding(texts) * math.sqrt(self.d_model)
        tgt = self.pos_encoder(tgt) # (B, Seq_txt, d_model)

        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        tgt_key_padding_mask = (texts == PAD_IDX).to(tgt.device)

        # 3. Трансформер
        out = self.transformer_decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        ) # (B, Seq_txt, d_model)

        # 4. Проекция в размер словаря
        logits = self.fc_out(out) # (B, Seq_txt, vocab_size)
        return logits

## Обучение модели
Обучаю модель на датасете

In [ ]:
!pip install wandb editdistance

In [ ]:
def calculate_split_ter(predictions, targets, idx2char, rus_letters_set):
    """Считает TER отдельно для всей строки и процент ошибок конкретно в русских буквах"""
    total_rus_errors = 0
    total_rus_count = 0

    for p, t in zip(predictions, targets):
        p_chars = [idx2char[idx] for idx in p.tolist() if idx not in (PAD_IDX, SOS_IDX, EOS_IDX)]
        t_chars = [idx2char[idx] for idx in t.tolist() if idx not in (PAD_IDX, SOS_IDX, EOS_IDX)]

        # Фильтруем только русские буквы из правильного ответа
        rus_targets = [c for c in t_chars if c in rus_letters_set]
        rus_preds = [c for c in p_chars if c in rus_letters_set]

        total_rus_errors += editdistance.eval(rus_preds, rus_targets)
        total_rus_count += len(rus_targets)

    rus_ter = total_rus_errors / max(total_rus_count, 1)
    return rus_ter

In [ ]:
# pip install wandb editdistance
import wandb
import editdistance
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import OneCycleLR # <--- 1. Импортируем шедулер

# Инициализируем сессию W&B
wandb.init(
    project="latex-ocr-mvp",
    config={
        "learning_rate": 3e-4, # <--- 2. Подняли базовый lr до 3e-4 для OneCycleLR
        "epochs": 20,
        "batch_size": 64,
        "vocab_size": VOCAB_SIZE,
        "max_len": 150
    }
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = OCRTransformer(vocab_size=VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(model.parameters(), lr=wandb.config.learning_rate)

# <--- 3. Инициализируем OneCycleLR сразу после оптимизатора и train_loader
scheduler = OneCycleLR(
    optimizer,
    max_lr=3e-4,
    steps_per_epoch=len(train_loader),
    epochs=wandb.config.epochs,
    pct_start=0.1
)

def calculate_ter(predictions, targets):
    """Считает Token Error Rate (расстояние Левенштейна на уровне токенов)"""
    total_distance = 0
    total_len = 0

    for p, t in zip(predictions, targets):
        p_list = p.tolist()
        t_list = t.tolist()

        if EOS_IDX in p_list: p_list = p_list[:p_list.index(EOS_IDX)]
        if EOS_IDX in t_list: t_list = t_list[:t_list.index(EOS_IDX)]

        total_distance += editdistance.eval(p_list, t_list)
        total_len += len(t_list)

    return total_distance / max(total_len, 1)

# --- Training Loop ---
for epoch in range(wandb.config.epochs):
    model.train()
    train_loss = 0.0

    for batch_idx, (images, texts) in enumerate(train_loader):
        images, texts = images.to(device), texts.to(device)

        decoder_input = texts[:, :-1]
        decoder_target = texts[:, 1:]

        optimizer.zero_grad()
        logits = model(images, decoder_input)

        loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
        loss.backward()

        # <--- 4. Добавляем клиппинг градиентов перед шагом оптимизатора
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # <--- 5. Шагаем шедулером на каждом батче (обязательно после optimizer.step())
        scheduler.step()

        train_loss += loss.item()

        if batch_idx % 10 == 0:
            wandb.log({"train/batch_loss": loss.item()})

    # --- Validation Loop ---
    model.eval()
    val_loss = 0.0
    val_ter = 0.0
    # --- Перед началом валидации создаем таблицу W&B ---
val_table = wandb.Table(columns=["Image", "True Text", "Pred Text"])

with torch.no_grad():
    for batch_idx, (images, texts) in enumerate(val_loader):
        images, texts = images.to(device), texts.to(device)
        decoder_input = texts[:, :-1]
        decoder_target = texts[:, 1:]

        logits = model(images, decoder_input)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
        val_loss += loss.item()

        predictions = torch.argmax(logits, dim=-1)
        val_ter += calculate_ter(predictions, decoder_target)

        # Логируем первые 5 картинок из первого батча в таблицу
        if batch_idx == 0:
            for i in range(min(5, len(images))):
                pred_text = "".join([idx2char[idx] for idx in predictions[i].tolist() if idx not in (PAD_IDX, SOS_IDX, EOS_IDX)])
                true_text = "".join([idx2char[idx] for idx in decoder_target[i].tolist() if idx not in (PAD_IDX, SOS_IDX, EOS_IDX)])

                # Денормализация картинки обратно в RGB для просмотра
                img_tensor = images[i].cpu() * 0.5 + 0.5
                img_pil = transforms.ToPILImage()(img_tensor)

                val_table.add_data(wandb.Image(img_pil), true_text, pred_text)


    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_val_ter = val_ter / len(val_loader)

    wandb.log({
        "epoch": epoch,
        "train/epoch_loss": avg_train_loss,
        "val/epoch_loss": avg_val_loss,
        "val/ter": avg_val_ter
    })

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val TER: {avg_val_ter:.4f}")

    wandb.log({"val/visual_predictions": val_table})

wandb.finish()

## Оценка модели по точности и времени
Сравнивую модель с тяжеловесным дипсиком по времени и точности ответов